In [1]:
"""
semantic_w2v.py
Запуск 1: статические эмбеддинги ruwikiruscorpora (word2vec).

Пересчитывает все метрики заново

Выходные файлы → output/semantic_w2v/
  per_context_word2vec_<model>.csv   — метрики по контекстам
  summary_word2vec_<model>.csv       — средние по top-k
"""
!pip install gensim
from pathlib import Path
import numpy as np
from gensim.models import KeyedVectors

from filter_data_llama_6 import get_all_models
from semantic_utils import load_data, prep_model_df, run_pipeline

model_wrd2vec = "model.bin"  # путь к файлу с моделью
output = Path("output/semantic_w2v")
output.mkdir(parents=True, exist_ok=True)  # создаем папку если ее нет

size_emb = 300  # размерность эмбеддингов

# ── Загружаем модель ──────────────────────────────────────────────────
print("Загружаем модель word2vec (ruwikiruscorpora)...")
kv = KeyedVectors.load_word2vec_format(model_wrd2vec, binary=True)
print(f"  В словаре {len(kv)} токенов")


def get_vec(lemma: str, upos: str, left_ctx: str = "") -> np.ndarray:
    """
    Получаем вектор для слова.
    left_ctx игнорируем - word2vec не видит контекст.
    """
    key = f"{lemma.lower()}_{upos.upper()}"  # ключ вида "дом_NOUN"

    # сначала ищем с тегом
    if key in kv:
        return kv[key].astype(float)

    # если нет - пробуем без тега
    if lemma.lower() in kv:
        return kv[lemma.lower()].astype(float)

    # если слова нет в словаре - возвращаем нулевой вектор
    return np.zeros(size_emb, dtype=float)


# Загружаем данные людей
print("\nЗагружаем данные...")
human_deduped, context_lookup = load_data()

# Запускаем для всех языковых моделей
for model_name, model_df in get_all_models().items():
    print(f"\n")
    print(f"модель: {model_name}")
    print(f"\n")

    # делаем slug для имени файла (без пробелов и тире)
    model_slug = model_name.replace(" ", "_").replace("-", "_")

    # подготавливаем данные модели
    model_deduped = prep_model_df(model_df)

    # проверяем сколько слов есть в словаре (OOV = out of vocabulary)
    n_found = sum(
        1 for lemma, upos in zip(model_deduped["lemma_word"],
                                  model_deduped["upos_word"])
        if get_vec(lemma, upos).any()  # any() вернет True если не нулевой
    )
    print(f"  Найдено в словаре: {n_found}/{len(model_deduped)} предсказаний")

    # запускаем пайплайн
    df = run_pipeline(
        approach_name="word2vec",
        get_vec=get_vec,
        human_deduped=human_deduped,
        model_deduped=model_deduped,
        context_lookup=context_lookup,
        out_dir=output,
        model_slug=model_slug,
    )

    # считаем средние метрики по top-k
    metric_cols = ["centroid_sim"] + [
        f"{m}_{t}"
        for t in [70, 80, 90]  # пороги 0.7, 0.8, 0.9
        for m in ("recall", "precision", "f1")
    ]

    # оставляем только те колонки, которые есть в датафрейме
    available = [c for c in metric_cols if c in df.columns]

    # группируем по top_k и считаем среднее
    summary = df.groupby("top_k")[available].mean().round(4).reset_index()

    # сохраняем сводку
    summary.to_csv(output / f"summary_word2vec_{model_slug}.csv", index=False)

    # выводим на экран
    print(f"\n  Сводка для {model_name}:")
    print("  " + summary.to_string(index=False))

print(f"\n закончили")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.8 MB/s eta 0:00:00
Загружаем модель word2vec (ruwikiruscorpora)...
  В словаре 249565 токенов

Загружаем данные...
Загружено людей: 2659 записей, 144 разных контекстов


модель: GPT-4o-mini


  Найдено в словаре: 16372/20144 предсказаний
  [word2vec] Найдено общих контекстов: 144
    Обработано 20/144...
    Обработано 40/144...
    Обработано 60/144...
    Обработано 80/144...
    Обработано 100/144...
    Обработано 120/144...
    Обработано 140/144...
  Сохранено в per_context_word2vec_GPT_4o_mini.csv (834 строк)

  Сводка для GPT-4o-mini:
   top_k  centroid_sim  recall_70  precision_70  f1_70  recall_80  precision_80  f1_80  recall_90  precision_90  f1_90
     1        0.5744     0.2315        0.6691 0.2998     0.2252        0.6691 0.2944     0.2251        0.6691 0.2941
     5        0.6879     0.3891        0.6256 0.4390     0.3709        0.6174 0.4237     0.3685        0.6123 0.4201
    10        0.6960     0.4028       